# Week 4 — Evaluation, Demo Polish, Full Metrics

**Inputs:** `models/lstm_ae.pt`, `models/isolation_forest.pkl`, `data/processed/test.parquet`

**Outputs:**
- Full metrics table (AUROC, F1, FPR) for both models
- Ablation: with/without `in_restricted_zone` feature
- PR curve plot
- `demo.py` wired to real model scores

**Deliverable:** End-to-end pipeline working on a laptop. Demo must run with no external calls.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/drone-ai-saturdays'
    !pip install -q torch scikit-learn pandas pyarrow matplotlib
else:
    BASE = '..'

import os
DATA_PROC = f'{BASE}/data/processed'
MODELS = f'{BASE}/models'

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve, roc_curve
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FEATURE_COLS = ['lat','lon','alt','speed','heading','dist_lemd','tod_sin','tod_cos']
WINDOW = 30

In [ ]:
# Load LSTM Autoencoder
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim=8, hidden_dim=64, n_layers=2):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, n_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, n_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, input_dim)
    def forward(self, x):
        _, (h, c) = self.encoder(x)
        latent = h[-1].unsqueeze(1).repeat(1, x.size(1), 1)
        decoded, _ = self.decoder(latent, (h, c))
        return self.output_layer(decoded)
    def anomaly_score(self, x):
        with torch.no_grad():
            return ((x - self(x)) ** 2).mean(dim=(1, 2))

lstm = LSTMAutoencoder().to(DEVICE)
lstm.load_state_dict(torch.load(f'{MODELS}/lstm_ae.pt', map_location=DEVICE))
lstm.eval()
threshold = float(np.load(f'{MODELS}/threshold.npy'))
print(f'LSTM loaded. Threshold: {threshold:.4f}')

# Load IF
with open(f'{MODELS}/isolation_forest.pkl', 'rb') as f:
    clf_if = pickle.load(f)
print('Isolation Forest loaded')

In [ ]:
# Build test set with injected anomalies
test_df = pd.read_parquet(f'{DATA_PROC}/test.parquet')

def inject_anomalies(df, n=300, seed=42):
    np.random.seed(seed)
    segs = df['seg_id'].unique()
    chosen = np.random.choice(segs, size=min(n, len(segs)), replace=False)
    anomalies = []
    for seg_id in chosen:
        seg = df[df['seg_id'] == seg_id].copy()
        atype = np.random.choice(['altitude', 'speed', 'hovering', 'zone_violation'])
        if atype == 'altitude':
            seg['alt'] = seg['alt'] + 3.0
        elif atype == 'speed':
            mid = len(seg) // 2
            seg.iloc[mid:mid+5, seg.columns.get_loc('speed')] *= 3
        elif atype == 'hovering':
            mid = len(seg) // 2
            seg.iloc[mid:mid+5, seg.columns.get_loc('speed')] = 0
        else:  # zone_violation — approach LEMD ARP directly
            seg['dist_lemd'] = seg['dist_lemd'] - 2.0  # 2 std devs closer
        seg['seg_id'] = seg_id + '_anom'
        seg['anomaly_type'] = atype
        anomalies.append(seg)
    return pd.concat(anomalies)

anomaly_df = inject_anomalies(test_df)
print(f'Normal: {test_df["seg_id"].nunique():,} segs, Anomalies: {anomaly_df["seg_id"].nunique():,} segs')

In [ ]:
# Score with LSTM
def score_lstm(df):
    scores = {}
    for seg_id, grp in df.groupby('seg_id'):
        vals = grp[FEATURE_COLS].values.astype(np.float32)
        if len(vals) < WINDOW:
            continue
        windows = [vals[i:i+WINDOW] for i in range(0, len(vals)-WINDOW+1, WINDOW//2)]
        x = torch.tensor(np.array(windows)).to(DEVICE)
        seg_scores = lstm.anomaly_score(x).cpu().numpy()
        scores[seg_id] = float(seg_scores.max())  # worst window = segment score
    return scores

normal_scores_lstm = score_lstm(test_df)
anomaly_scores_lstm = score_lstm(anomaly_df)
print(f'Scored {len(normal_scores_lstm)} normal + {len(anomaly_scores_lstm)} anomaly segs')

In [ ]:
# Score with Isolation Forest
def traj_stats(df):
    return df.groupby('seg_id')[FEATURE_COLS].agg(['mean','std','min','max']).fillna(0)

X_normal  = traj_stats(test_df)
X_anomaly = traj_stats(anomaly_df)
if_normal_scores  = -clf_if.decision_function(X_normal)
if_anomaly_scores = -clf_if.decision_function(X_anomaly)
print('IF scored')

In [ ]:
def compute_metrics(normal_scores, anomaly_scores, threshold=None):
    y = np.array([0]*len(normal_scores) + [1]*len(anomaly_scores))
    scores = np.concatenate([normal_scores, anomaly_scores])
    auroc = roc_auc_score(y, scores)
    if threshold is None:
        threshold = np.percentile(normal_scores, 95)
    preds = (scores > threshold).astype(int)
    f1 = f1_score(y, preds, zero_division=0)
    fpr = (preds[:len(normal_scores)]).sum() / len(normal_scores)
    return auroc, f1, fpr, threshold

lstm_scores_arr = np.array([normal_scores_lstm.get(k, 0) for k in test_df['seg_id'].unique()])
lstm_anom_arr   = np.array([anomaly_scores_lstm.get(k, 0) for k in anomaly_df['seg_id'].unique()])

lstm_auroc, lstm_f1, lstm_fpr, _ = compute_metrics(lstm_scores_arr, lstm_anom_arr, threshold)
if_auroc, if_f1, if_fpr, _       = compute_metrics(if_normal_scores, if_anomaly_scores)

print('| Model | AUROC | F1 | FPR |')
print('|---|---|---|---|')
print(f'| LSTM Autoencoder | {lstm_auroc:.3f} | {lstm_f1:.3f} | {lstm_fpr:.3f} |')
print(f'| Isolation Forest | {if_auroc:.3f} | {if_f1:.3f} | {if_fpr:.3f} |')

# Success criteria check
print()
print('SUCCESS CRITERIA:')
print(f'  AUROC > 0.85: {"PASS" if lstm_auroc > 0.85 else "FAIL"} ({lstm_auroc:.3f})')
print(f'  FPR <= 0.15:  {"PASS" if lstm_fpr <= 0.15 else "FAIL"} ({lstm_fpr:.3f})')
print(f'  IF < LSTM:    {"PASS" if if_auroc < lstm_auroc else "FAIL"} (IF {if_auroc:.3f} vs LSTM {lstm_auroc:.3f})')

In [ ]:
# Precision-recall curve
y_full = np.array([0]*len(lstm_scores_arr) + [1]*len(lstm_anom_arr))
scores_full = np.concatenate([lstm_scores_arr, lstm_anom_arr])
precision, recall, _ = precision_recall_curve(y_full, scores_full)

plt.figure(figsize=(6,4))
plt.plot(recall, precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title(f'LSTM Autoencoder PR Curve (AUROC={lstm_auroc:.3f})')
plt.tight_layout()
plt.savefig(f'{BASE}/docs/weekly/figures/week4_pr_curve.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Ablation: drop dist_lemd (proxy for restricted zone feature)
ABLATION_COLS = [c for c in FEATURE_COLS if c != 'dist_lemd']
print('Ablation: model without dist_lemd (restricted zone proxy)')
print('(Re-train required for full ablation — this is a placeholder structure)')
print('Record delta AUROC in writeup Table 2.')